# 🌊 AquaSentinel AI — Model Training & Evaluation Guide
### Autonomous Underwater Target Detection in Side-Scan & Multibeam Sonar Imagery

Welcome to the **AquaSentinel AI** model training guide. This interactive Jupyter Notebook (`training.ipynb`) is designed to onboard new project members, researchers, and hydrographic survey engineers so they can seamlessly reproduce, audit, and evaluate the exact deep learning model training workflow used in this repository.

---

### 📌 Very Important: Single Source of Truth
> **This notebook does NOT duplicate or rewrite the training algorithms.**  
> The production training logic resides exclusively in [`ml/train.py`](file:///d:/Projects/SIH_2026/ml/train.py) and evaluation logic in [`ml/evaluate.py`](file:///d:/Projects/SIH_2026/ml/evaluate.py). This notebook acts as an interactive command interface, orchestrator, and educational walkthrough. Any future updates made to `ml/train.py` are automatically reflected whenever this notebook is run.

---

### 🎯 What This Project & Model Does
* **Domain**: Hydrographic Acoustic Remote Sensing (Side-Scan Sonar & Forward-Looking Sonar).
* **Architecture**: Ultralytics **YOLOv8n / YOLO11** optimized with custom acoustic backscatter augmentations (intensity gain invariance, port/starboard track symmetry, and shadow contrast validation).
* **Input Data**: 640×640 single-channel acoustic backscatter sonar tiles.
* **Output Predictions**: Target bounding boxes, acoustic category classifications, confidence scores, and physical hazard ratings (Critical, High, Medium, Low).
* **Downstream Integration**: The trained weights (`MODELS/yolov8n.pt`) are automatically consumed by the FastAPI backend (`backend/inference.py`) and visualized live on the React mission dashboard.

---

### 🗺️ Quick Start Roadmap
1. **Understand the Project Structure**: Review key directories (`FINAL_DATASET`, `ml`, `MODELS`, `runs`, `Test_Data`).
2. **Review User Configuration**: Verify project paths and hyperparameters.
3. **Check Guidelines**: Learn what settings should NOT be modified.
4. **Inspect Environment**: Verify Python, PyTorch, CUDA GPU, and VRAM.
5. **Install Dependencies**: Ensure required packages are up to date.
6. **Verify Dataset Integrity**: Audit train, validation, and test image/label counts.
7. **Inspect Dataset Config**: Review class labels and YAML configuration (`dataset.yaml`).
8. **Launch Training Pipeline**: Run `ml/train.py` through the existing script.
9. **Locate Model Checkpoints**: Confirm trained weights are exported to `MODELS/yolov8n.pt`.
10. **Validate on Test Benchmark**: Benchmark generalization with `ml/evaluate.py`.
11. **Run Inference**: Predict on sample sonar images and visualize detections.
12. **Review Performance Graphs**: Inspect loss curves, PR curves, and confusion matrix.


---
## Section 1 — Project Structure

Here is the actual layout of the **AquaSentinel / Commit-Crack** repository:

```text
Commit-Crack/
├── FINAL_DATASET/               # Unified side-scan sonar dataset (2.1+ GB)
│   ├── dataset.yaml             # 5-Class dataset configuration file
│   ├── images/                  # Acoustic backscatter sonar tiles (640x640)
│   │   ├── train/               # 5,889 Training images (70%)
│   │   ├── val/                 # 954 Validation images (15%)
│   │   └── test/                # Unseen benchmark test split (15%)
│   └── labels/                  # YOLO format .txt annotations (class_id xc yc w h)
│       ├── train/               # 5,889 Training label files
│       ├── val/                 # 954 Validation label files
│       └── test/                # Unseen benchmark test label files
│
├── MODELS/                      # Dedicated production model weights directory
│   └── yolov8n.pt               # Fine-tuned 5-class neural network weights (22.5 MB)
│
├── ml/                          # Core machine learning source code (Source of Truth)
│   ├── train.py                 # Primary training script with GPU/CPU hardware partition
│   ├── evaluate.py              # Test split benchmark evaluation script
│   ├── prepare_dataset.py       # Dataset audit, normalization, and split generator
│   ├── class_map.json           # Taxonomy mapping (hazard levels, distractor tags)
│   ├── model_info.json          # Architecture specifications and benchmark metadata
│   └── samples/                 # Sample sonar images for testing
│
├── backend/                     # High-performance FastAPI Python inference backend
│   ├── main.py                  # REST API endpoints (GET /health, POST /analyze)
│   ├── inference.py             # Full survey pipeline (EGN, CLAHE, Slant-Range, AHSD)
│   ├── models.py                # Pydantic schemas mirroring frontend types
│   ├── pipeline/                # Modules: geolocation, shadow gate, slant range, tiling
│   └── requirements.txt         # Production backend dependencies
│
├── frontend/                    # React 18 + Vite + TypeScript tactical dashboard
│   ├── src/                     # Sonar image visualizer, target drawer, mission map
│   └── package.json             # Frontend UI dependencies
│
├── runs/                        # Training outputs, loss curves, confusion matrices
│   └── detect/runs/train/aquasentinel_training/
│       ├── weights/best.pt      # Best epoch checkpoint based on validation loss
│       ├── results.png          # Box loss, class loss, DFL loss, and mAP curves
│       ├── confusion_matrix.png # Confusion matrix across all classes
│       └── val_batch0_pred.jpg  # Visual predictions on validation batches
│
├── Test_Data/                   # Real-world authentic sonar mission imagery (.jpg / .png)
├── start.bat                    # 1-Click Windows launcher (starts FastAPI + Vite)
├── README.md                    # Project documentation
└── training.ipynb               # This training walkthrough notebook
```


---
## Section 2 — Single Source of Truth Architecture

To prevent code duplication, version drift, and synchronization bugs, this project adheres to a strict **Single Source of Truth** pipeline architecture:

```text
       ┌─────────────────────────────────────────────────────────┐
       │             ml/train.py  (Main Training Code)           │
       │  • Hardware-aware GPU allocation (Ampere TF32 & FP16)   │
       │  • Sonar physics augmentations (No hue distortion)      │
       │  • Windows native streaming (workers=0 direct stream)   │
       │  • Auto-export to MODELS/yolov8n.pt                     │
       └────────────────────────────┬────────────────────────────┘
                                    │
                  Executes via shell/CLI invocation
                                    │
                                    ▼
       ┌─────────────────────────────────────────────────────────┐
       │          training.ipynb  (Interface & Guide)            │
       │  • Pre-flight diagnostics & environment validation       │
       │  • Dataset integrity checks                             │
       │  • Calls ml/train.py and ml/evaluate.py                 │
       │  • Visualizes outputs, loss curves & confusion matrix   │
       └─────────────────────────────────────────────────────────┘
```

> [!TIP]
> If you make improvements or changes to hyperparameters or model architectures in `ml/train.py`, you **do not need to rewrite anything in this notebook**. Re-running the training cell in this notebook will immediately use your latest code.


---
## Section 3 — User Configuration

### Markdown Explanation
* **What this command does**: Configures the project-relative paths, hardware selections, and training parameters for this interactive session.
* **Why we are running it**: To establish clean relative references rather than machine-specific absolute paths, ensuring this notebook runs cleanly on any developer's machine.
* **What output to expect**: Prints the resolved project paths and current hyperparameter configuration.
* **What you can safely change**:
  * `EPOCHS`: Change between `15` (quick smoke test) and `50` (full convergence).
  * `BATCH_SIZE`: `16` is ideal for 4 GB GPUs (e.g. RTX 3050). If you experience Out Of Memory (OOM), reduce to `8`. If you have an 8+ GB GPU, you can increase to `32`.
  * `DEVICE`: Use `"0"` for NVIDIA CUDA GPU training, or `"cpu"` for CPU mode.
  * `SAMPLE_IMAGE`: Point to any sonar image inside `Test_Data/` to test inference.
* **What you should normally leave unchanged**:
  * `PROJECT_ROOT`, `DATASET_PATH`, `BASE_MODEL`, and `EXPORT_PATH`.


In [ ]:
# ==============================================================================
# USER CONFIGURATION: Adjust these settings as needed for your local machine
# ==============================================================================
import os
import sys
from pathlib import Path

# Project root directory (project-relative, resolved dynamically)
PROJECT_ROOT = Path(".").resolve()

# Path to the dataset configuration YAML
DATASET_PATH = PROJECT_ROOT / "FINAL_DATASET" / "dataset.yaml"

# Starting baseline model weights (fine-tune from existing checkpoint or base yolo)
BASE_MODEL = PROJECT_ROOT / "MODELS" / "yolov8n.pt"

# Training Hyperparameters
EPOCHS = 50           # Total training passes (50 recommended; use 10-15 for a quick test)
BATCH_SIZE = 16       # Batch size: 16 (optimized for 4GB VRAM). Set to 8 if OOM occurs.
IMAGE_SIZE = 640       # Resolution: 640x640 pixels (standard for sonar tiles)
DEVICE = "0"          # '0' for CUDA GPU #0; use 'cpu' only if no dedicated GPU exists
PATIENCE = 15         # Early stopping patience (stop if no improvement for 15 epochs)

# Destination to export best performing weights for backend use
EXPORT_PATH = PROJECT_ROOT / "MODELS" / "yolov8n.pt"

# Sample sonar image for post-training inference testing
SAMPLE_IMAGE = PROJECT_ROOT / "Test_Data" / "pipe_1693569563.809_x3000.jpg"

print("============================================================")
print("             ACTIVE CONFIGURATION SETTINGS")
print("============================================================")
print(f" Project Root:     {PROJECT_ROOT}")
print(f" Dataset YAML:     {DATASET_PATH} (Found: {DATASET_PATH.exists()})")
print(f" Baseline Weights: {BASE_MODEL} (Found: {BASE_MODEL.exists()})")
print(f" Export Target:    {EXPORT_PATH}")
print(f" Target Epochs:    {EPOCHS}")
print(f" Batch Size:       {BATCH_SIZE}")
print(f" Compute Device:   '{DEVICE}'")
print(f" Sample Image:     {SAMPLE_IMAGE.name} (Found: {SAMPLE_IMAGE.exists()})")
print("============================================================")


---
## Section 4 — Guidelines on What NOT to Change

> [!WARNING]
> Please read this carefully before proceeding. Modifying these core elements will break downstream compatibility with the FastAPI backend and the React UI.

1. **Do NOT Alter the 5 Canonical Class IDs or Names**:
   The entire system (frontend telemetry cards, risk calculation, acoustic shadow filters) depends on this exact taxonomy:
   * `0: crab_pot` (Hard negative / distractor)
   * `1: submarine_pipeline` (Active production class)
   * `2: shipwreck` (Active production class)
   * `3: ghost_net` (Active production class)
   * `4: mine_cylinder` (Critical safety production class)
2. **Do NOT Change YOLO Annotation Format**:
   Labels must remain normalized YOLO format (`<class_id> <x_center> <y_center> <width> <height>` between `0.0` and `1.0`).
3. **Do NOT Change Input Resolution**:
   The input dimension must remain **640×640**. Sonar beam physics and slant-range normalization in the pipeline assume this tile geometry.
4. **Do NOT Enable Hue/Saturation Augmentations**:
   Sonar imagery is single-channel acoustic backscatter. Artificial hue shifts violate underwater acoustics and degrade model performance.


---
## Section 5 — Environment Diagnostics

### Markdown Explanation
* **What this command does**: Inspects your Python runtime, PyTorch framework installation, CUDA driver availability, and GPU hardware specifications.
* **Why we are running it**: To verify that your machine has hardware acceleration enabled. Training on an NVIDIA GPU is **25× to 30× faster** than CPU training.
* **What output to expect**:
  * Python version (Python 3.10+ recommended)
  * PyTorch version
  * Ultralytics YOLO version
  * CUDA availability (`True` if an NVIDIA GPU is configured)
  * Dedicated VRAM in Gigabytes
* **Whether you need to change anything**: If `CUDA Available` returns `False`, see the instructions in the output to install the CUDA-enabled PyTorch build.


In [ ]:
import sys
import torch
import ultralytics

print("============================================================")
print("             ENVIRONMENT HARDWARE DIAGNOSTICS")
print("============================================================")
print(f" Python Version:      {sys.version.split()[0]} ({sys.executable})")
print(f" PyTorch Version:     {torch.__version__}")
print(f" Ultralytics Version: {ultralytics.__version__}")
print(f" CUDA Available:      {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_id = 0
    gpu_name = torch.cuda.get_device_name(gpu_id)
    vram_gb = torch.cuda.get_device_properties(gpu_id).total_memory / (1024 ** 3)
    cuda_cap = torch.cuda.get_device_capability(gpu_id)
    print(f" GPU Device [{gpu_id}]:       {gpu_name}")
    print(f" Dedicated VRAM:      {vram_gb:.2f} GB")
    print(f" Compute Capability:  {cuda_cap[0]}.{cuda_cap[1]}")
    print(" status:              CUDA GPU acceleration is READY.")
else:
    print(" status:              CUDA is NOT detected. (Running on CPU)")
    print(" Note: If your laptop has an NVIDIA RTX GPU, install the CUDA build:")
    print(" pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu126")
print("============================================================")


---
## Section 6 — Install Dependencies

### Markdown Explanation
* **What this command does**: Installs or verifies all required Python libraries specified in [`backend/requirements.txt`](file:///d:/Projects/SIH_2026/backend/requirements.txt) plus plotting tools (`matplotlib`, `pyyaml`, `pandas`).
* **Why we are running it**: To guarantee that Ultralytics, OpenCV, PyTorch, PyYAML, and Pillow are installed with compatible versions.
* **What output to expect**: `Requirement already satisfied` or progress bars showing downloaded wheels.
* **Whether you need to change anything**: Usually nothing. If you are already running in the project virtual environment, this will quickly confirm all packages are present.


In [ ]:
# Install project dependencies from existing backend/requirements.txt
!pip install -r backend/requirements.txt pyyaml matplotlib pandas

print("\n[SUCCESS] Dependencies verified.")


---
## Section 7 — Check Dataset Integrity

### Markdown Explanation
* **What this command does**: Performs a pre-flight audit of the dataset directory (`FINAL_DATASET`). It counts image files (`.jpg`, `.png`) and label files (`.txt`) across the `train`, `val`, and `test` splits.
* **Why we are running it**: Catching a missing dataset path, missing images, or mismatched labels before starting training prevents silent crashes or hours of wasted compute.
* **What output to expect**:
  * Training Images count (~5,889) and Training Labels count (~5,889)
  * Validation Images count (~954) and Validation Labels count (~954)
  * Test Images count (~300+)
  * Total image count
* **Whether you need to change anything**: If counts are 0, check that `FINAL_DATASET` is located at `PROJECT_ROOT / "FINAL_DATASET"`.


In [ ]:
from pathlib import Path

dataset_dir = PROJECT_ROOT / "FINAL_DATASET"

# Collect image and label lists across splits
train_imgs = list((dataset_dir / "images" / "train").glob("*.*"))
val_imgs = list((dataset_dir / "images" / "val").glob("*.*"))
test_imgs = list((dataset_dir / "images" / "test").glob("*.*"))

train_lbls = list((dataset_dir / "labels" / "train").glob("*.txt"))
val_lbls = list((dataset_dir / "labels" / "val").glob("*.txt"))
test_lbls = list((dataset_dir / "labels" / "test").glob("*.txt"))

total_imgs = len(train_imgs) + len(val_imgs) + len(test_imgs)
total_lbls = len(train_lbls) + len(val_lbls) + len(test_lbls)

print("============================================================")
print("             DATASET INTEGRITY PRE-FLIGHT AUDIT")
print("============================================================")
print(f" Dataset Root:      {dataset_dir.resolve()}")
print(f" Training Split:    {len(train_imgs):>5} images  | {len(train_lbls):>5} labels")
print(f" Validation Split:  {len(val_imgs):>5} images  | {len(val_lbls):>5} labels")
print(f" Testing Split:     {len(test_imgs):>5} images  | {len(test_lbls):>5} labels")
print("------------------------------------------------------------")
print(f" Total Dataset:     {total_imgs:>5} images  | {total_lbls:>5} labels")
print("============================================================")

assert len(train_imgs) > 0, "[ERROR] No training images found in FINAL_DATASET/images/train!"
assert len(val_imgs) > 0, "[ERROR] No validation images found in FINAL_DATASET/images/val!"
print("[PASSED] Dataset files successfully verified!")


---
## Section 8 — Explain Dataset Configuration (`dataset.yaml`)

### Markdown Explanation
* **What this command does**: Loads and prints the project's dataset configuration file ([`FINAL_DATASET/dataset.yaml`](file:///d:/Projects/SIH_2026/FINAL_DATASET/dataset.yaml)).
* **Why we are running it**: To verify the paths to the image splits and review the 5 canonical classes.
* **Fields explained**:
  * `path`: Base directory for all image splits.
  * `train`: Relative path to training images (`images/train`).
  * `val`: Relative path to validation images (`images/val`).
  * `test`: Relative path to unseen test images (`images/test`).
  * `nc`: Number of object classes (5).
  * `names`: Dictionary mapping class index to human-readable label.
* **Important Note**: Do NOT manually edit this file. The class ordering must remain consistent.


In [ ]:
import yaml

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    yaml_config = yaml.safe_load(f)

print("============================================================")
print("             DATASET YAML CONFIGURATION")
print("============================================================")
print(f" Path:         {yaml_config.get('path')}")
print(f" Train Split:  {yaml_config.get('train')}")
print(f" Val Split:    {yaml_config.get('val')}")
print(f" Test Split:   {yaml_config.get('test')}")
print(f" Num Classes:  {yaml_config.get('nc')}")
print("------------------------------------------------------------")
print(" Class Taxonomy Mapping:")
for cid, cname in yaml_config.get("names", {}).items():
    role = "Distractor (Hard Negative)" if int(cid) == 0 else "Active Production Class"
    print(f"   [{cid}] {cname:22s} -> {role}")
print("============================================================")


---
### Deep Dive: The 5 Canonical Hydrographic Sonar Classes

| Class ID | Class Label | Role & Function | Operational Significance |
|:---|:---|:---|:---|
| `0` | **`crab_pot`** | **Hard Negative / Distractor** | Man-made crab/lobster traps and seabed fixtures. The model is trained on these to *suppress false alarms* in the mission dashboard. |
| `1` | **`submarine_pipeline`** | **Production Class** | Subsea oil/gas pipelines, buried trunklines, and exposed communications cables. |
| `2` | **`shipwreck`** | **Production Class** | Wrecked vessels, sunken barges, airplane wreckage, and anomalous structural debris. |
| `3` | **`ghost_net`** | **Production Class** | Abandoned/lost fishing nets and derelict fishing gear that entangle marine wildlife. |
| `4` | **`mine_cylinder`** | **Critical Safety Class** | Cylindrical acoustic targets, discarded oil drums, and suspected underwater ordnance (UXO). |


---
## Section 9 — Training Settings & Hyperparameters

Below are the exact hyperparameters used in the local training pipeline:

* **Epochs (`50`)**: One epoch is one complete pass through the 5,889 training images. 50 epochs provides strong convergence without overfitting.
* **Batch Size (`16`)**: Number of sonar images processed in parallel per GPU forward/backward pass. 16 is mathematically optimized to utilize ~3.6 GB of VRAM on a 4 GB GPU (NVIDIA RTX 3050).
* **Resolution (`640×640`)**: Standard multibeam / side-scan sonar tile dimensions.
* **Optimizer (`auto`)**: Autoselected by Ultralytics (AdamW or SGD with momentum 0.937, weight decay 0.0005).
* **Learning Rate (`lr0=0.01`, `lrf=0.01`)**: Initial learning rate 0.01 with cosine / linear learning rate scheduling down to final multiplier 0.01.
* **AMP (`amp=True`)**: Automatic Mixed Precision (FP16) enabled for Tensor Core acceleration on Ampere GPUs.
* **Workers (`workers=0`)**: Direct single-process GPU stream on Windows to avoid Python multiprocessing pickling errors (`OSError 22`).
* **Sonar Acoustic Augmentations**:
  * `fliplr=0.5`: Horizontally flips sonar tiles (port and starboard sonar acoustic tracks are physically symmetric).
  * `degrees=10.0`: Simulates slight vehicle yaw angle and underwater current drift.
  * `mosaic=0.5`: Stitches 4 acoustic patches together to expose the model to diverse seafloor textures.
  * `hsv_h=0.0` & `hsv_s=0.0`: **Hue and saturation augmentation is explicitly disabled (0.0)** because sonar is single-channel acoustic backscatter intensity.
  * `hsv_v=0.2`: Simulates Time-Varied Gain (TVG) intensity variations across range bins.


---
## Section 10 — Start Model Training

### Markdown Explanation
* **What this command does**: Calls the existing project training script [`ml/train.py`](file:///d:/Projects/SIH_2026/ml/train.py) with the parameters defined above.
* **Why we are running it**: To train the YOLO detection model using the project's official, single-source-of-truth training pipeline.
* **What output to expect**:
  1. Dataset Pre-flight Audit (verifying file counts)
  2. Hardware Configuration Summary (GPU details, VRAM, and Tensor Core status)
  3. Real-time epoch logs displaying Box Loss, Class Loss, DFL Loss, and validation mAP
  4. Post-training checkpoint export confirmation (`MODELS/yolov8n.pt`)
  5. Automated post-training validation benchmark on the unseen test split
* **Whether you need to change anything**:
  * If training with GPU: Run the cell as is (`--device 0`).
  * If training on CPU: Add the `--allow_cpu` flag to the command.


In [ ]:
# ==============================================================================
# MODEL TRAINING CELL: Calls the existing project training script (ml/train.py)
# ==============================================================================

# Construct the training command using active configuration
cmd = (
    f"python ml/train.py "
    f"--data \"{DATASET_PATH}\" "
    f"--model \"{BASE_MODEL}\" "
    f"--epochs {EPOCHS} "
    f"--batch {BATCH_SIZE} "
    f"--imgsz {IMAGE_SIZE} "
    f"--device {DEVICE} "
    f"--patience {PATIENCE} "
    f"--export \"{EXPORT_PATH}\""
)

# Note: If running on a machine without a dedicated CUDA GPU, append: --allow_cpu
if DEVICE == "cpu":
    cmd += " --allow_cpu"

print("Executing Training Command:")
print(f"  {cmd}\n")

# Run the project source training script
!{cmd}


---
## Section 11 — Explain What Happens During Training

When training is running, Ultralytics outputs a formatted progress table for every epoch:

```text
      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      3.22G      1.452      1.120      1.312         42        640: 100%|██████████|
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)
                   all        954        982      0.820      0.624      0.658      0.493
```

### What These Metrics Mean:
* **`box_loss`**: Bounding Box Regression Loss (CIoU loss). Measures how accurately the predicted bounding boxes overlap with ground truth annotations. Lower is better.
* **`cls_loss`**: Classification Loss (Binary Cross-Entropy). Measures whether the model correctly identifies the target category (e.g. `submarine_pipeline` vs `crab_pot`). Lower is better.
* **`dfl_loss`**: Distribution Focal Loss. Helps the network predict fine-grained bounding box boundaries in noisy acoustic returns. Lower is better.
* **`Box(P)` (Precision)**: Out of all detections made, what percentage were real underwater targets? High precision means low false alarms.
* **`R` (Recall)**: Out of all real targets in the seabed, what percentage did the model find? High recall means few missed targets.
* **`mAP50`**: Mean Average Precision at IoU threshold 0.50. Standard benchmark for object detection.
* **`mAP50-95`**: Mean Average Precision averaged across IoU thresholds from 0.50 to 0.95. The strictest metric for localization quality.
* **Checkpoints**: Every 5 epochs (`save_period=5`), an intermediate checkpoint is saved (`epoch5.pt`, `epoch10.pt`, etc.). The best model based on validation fitness is saved to `best.pt`.


---
## Section 12 — Find the Trained Model & Checkpoints

### Markdown Explanation
* **What this command does**: Inspects the filesystem to verify that the trained weights were saved and exported to the production directory ([`MODELS/yolov8n.pt`](file:///d:/Projects/SIH_2026/MODELS/yolov8n.pt)).
* **Why we are running it**: To confirm that the model checkpoint was successfully generated, check its file size (should be ~22.5 MB), and ensure it is ready for deployment.
* **What output to expect**: Confirms `MODELS/yolov8n.pt` exists, along with the raw training checkpoint in `runs/.../weights/best.pt`.


In [ ]:
import datetime
from pathlib import Path

# Paths where weights are saved and exported
target_checkpoints = [
    PROJECT_ROOT / "MODELS" / "yolov8n.pt",
    PROJECT_ROOT / "runs" / "detect" / "runs" / "train" / "aquasentinel_training" / "weights" / "best.pt",
    PROJECT_ROOT / "runs" / "train" / "aquasentinel_training" / "weights" / "best.pt",
    PROJECT_ROOT / "models" / "best.pt"
]

print("============================================================")
print("             TRAINED MODEL CHECKPOINT AUDIT")
print("============================================================")
for cp in target_checkpoints:
    if cp.exists():
        size_mb = cp.stat().st_size / (1024 * 1024)
        mtime = datetime.datetime.fromtimestamp(cp.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
        print(f" [FOUND] {cp}")
        print(f"         Size: {size_mb:.2f} MB | Last Modified: {mtime}")
    else:
        print(f" [NOT FOUND] {cp}")
print("============================================================")


---
## Section 13 — Validation / Evaluation on Unseen Test Split

### Markdown Explanation
* **What this command does**: Calls the project's existing evaluation script [`ml/evaluate.py`](file:///d:/Projects/SIH_2026/ml/evaluate.py) to benchmark the trained model on the unseen `test` split.
* **Why we are running it**: To evaluate true generalization performance on sonar imagery that was completely held out during training.
* **What output to expect**:
  * Overall test split mAP50 and mAP50-95
  * Per-class performance breakdown for each of the 5 hydrographic classes
  * Average inference latency per image in milliseconds
* **Whether you need to change anything**: You can change `--device` or `--model` if evaluating a different checkpoint.


In [ ]:
# Execute the project source evaluation script (ml/evaluate.py)
eval_cmd = (
    f"python ml/evaluate.py "
    f"--model \"{EXPORT_PATH}\" "
    f"--data \"{DATASET_PATH}\" "
    f"--imgsz {IMAGE_SIZE} "
    f"--device {DEVICE}"
)

print("Executing Evaluation Command:")
print(f"  {eval_cmd}\n")

!{eval_cmd}


---
## Section 14 — Testing / Inference on Sample Sonar Imagery

### Markdown Explanation
* **What this command does**: Runs object detection inference on a real-world test sonar image from `Test_Data/` using the trained weights (`MODELS/yolov8n.pt`).
* **Why we are running it**: To visually inspect how the model detects underwater targets, boxes acoustic highlights and shadows, and outputs confidence scores.
* **What output to expect**:
  * Text output displaying number of detections, predicted class label, confidence percentage, and bounding box coordinates.
  * An annotated plot displayed directly in the notebook showing the bounding box on top of the sonar scan.
* **Whether you need to change anything**: You can set `SAMPLE_IMAGE` to any other image inside `Test_Data/` (e.g. `Test_Data/0001_2010.jpg`, `Test_Data/synth_ghost_net_00009.png`).


In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Verify test image exists
test_img_path = Path(SAMPLE_IMAGE)
assert test_img_path.exists(), f"Sample image not found: {test_img_path}"

# Load trained model
print(f"Loading trained model: {EXPORT_PATH}")
model = YOLO(str(EXPORT_PATH))

# Run inference
print(f"Running inference on: {test_img_path.name}")
results = model.predict(source=str(test_img_path), conf=0.25, save=False)
result = results[0]

# Display predictions
print("\n" + "=" * 60)
print(f"INFERENCE RESULTS: {len(result.boxes)} target(s) detected")
print("=" * 60)
for idx, box in enumerate(result.boxes):
    cls_id = int(box.cls[0])
    cls_name = model.names[cls_id]
    conf = float(box.conf[0])
    coords = [round(float(c), 1) for c in box.xyxy[0].tolist()]
    print(f" Target #{idx + 1}: Class: '{cls_name}' (ID: {cls_id}) | Confidence: {conf * 100:.1f}% | Box: {coords}")
print("=" * 60)

# Render annotated image in notebook
annotated_bgr = result.plot()
annotated_rgb = annotated_bgr[..., ::-1]  # BGR to RGB

plt.figure(figsize=(10, 8))
plt.imshow(annotated_rgb)
plt.axis("off")
plt.title(f"Acoustic Target Detection: {test_img_path.name}", fontsize=14, pad=10)
plt.tight_layout()
plt.show()


---
## Section 15 — Training Results & Visualization

### Markdown Explanation
* **What this command does**: Loads and displays the training graphs automatically generated by the training pipeline:
  1. **`results.png`**: Training and validation loss curves (Box, Class, DFL) and mAP50 / mAP50-95 progression over all 50 epochs.
  2. **`confusion_matrix_normalized.png`**: Multi-class confusion matrix showing accuracy across the 5 canonical classes.
  3. **`val_batch0_pred.jpg`**: Visual ground truth vs prediction comparison on validation batches.
  4. **`results.csv`**: Numerical metrics table for the final epoch.
* **Why we are running it**: Visual analysis of training curves confirms healthy learning dynamics, absence of catastrophic overfitting, and strong class discrimination.


In [ ]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def render_plot(title: str, pattern: str, figsize=(14, 9)):
    """Helper to search runs/ directory and render image."""
    matches = list(PROJECT_ROOT.glob(pattern))
    if matches:
        img_path = matches[0]
        img = Image.open(img_path)
        plt.figure(figsize=figsize)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"{title} ({img_path.name})", fontsize=14, pad=10)
        plt.tight_layout()
        plt.show()
    else:
        print(f"[NOTE] Artifact not found matching pattern: {pattern}")

# 1. Training Curves (Losses & mAP Progression)
render_plot("Training Loss & Metric Convergence", "runs/**/aquasentinel_training/results.png")

# 2. Multi-Class Confusion Matrix (Normalized)
render_plot("Normalized Confusion Matrix", "runs/**/aquasentinel_training/confusion_matrix_normalized.png", figsize=(10, 8))

# 3. Validation Batch Predictions
render_plot("Validation Predictions vs Ground Truth", "runs/**/aquasentinel_training/val_batch0_pred.jpg", figsize=(14, 9))

# 4. Final Epoch Numerical Metrics from results.csv
results_csvs = list(PROJECT_ROOT.glob("runs/**/aquasentinel_training/results.csv"))
if results_csvs:
    df = pd.read_csv(results_csvs[0])
    df.columns = [c.strip() for c in df.columns]
    print("\n" + "=" * 60)
    print("FINAL CONVERGED EPOCH METRICS (results.csv)")
    print("=" * 60)
    final_epoch = df.iloc[-1]
    for metric, value in final_epoch.items():
        if isinstance(value, float):
            print(f" {metric:26s}: {value:.6f}")
        else:
            print(f" {metric:26s}: {value}")
    print("=" * 60)


---
## Section 16 — Troubleshooting Guide for New Members

If you encounter an issue when training or evaluating the model, refer to these common solutions:

### 1. Dataset Not Found (`FINAL_DATASET` does not exist)
* **Symptom**: `[ERROR] Dataset configuration file not found at: ...`
* **Fix**: Ensure `FINAL_DATASET` is located at the project root (`Commit-Crack/FINAL_DATASET`). If your dataset is stored in another directory or external SSD, update `DATASET_PATH` in the **User Configuration** cell.

### 2. GPU Not Detected / CUDA Unavailable
* **Symptom**: `CUDA Available: False` or `[STRICT GPU POLICY ACTIVATED] CPU TRAINING IS DISABLED`.
* **Fix**: Your Python environment likely has the CPU-only build of PyTorch installed. Run this command in your terminal to install PyTorch with CUDA 12.6 support:
  ```powershell
  pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu126
  ```

### 3. GPU Out of Memory (CUDA OOM)
* **Symptom**: `RuntimeError: CUDA out of memory. Tried to allocate ...`
* **Fix**: Reduce the `BATCH_SIZE` in the **User Configuration** cell from `16` to `8`. You can also ensure no background applications (like games or video editors) are consuming GPU VRAM.

### 4. Windows Multiprocessing Pickling Error (`OSError: [Errno 22] Invalid argument`)
* **Symptom**: Crashes when starting epochs with `_MultiProcessingDataLoaderIter`.
* **Fix**: PyTorch on Windows uses process spawning that can conflict with dataset serialization. `ml/train.py` automatically sets `workers=0` on Windows, which routes data directly into GPU memory without multi-process IPC serialization issues.

### 5. Wrong Current Working Directory
* **Symptom**: `FileNotFoundError` for `ml/train.py` or `backend/requirements.txt`.
* **Fix**: Run `import os; print(os.getcwd())`. The working directory must be the project root directory (`Commit-Crack` or `SIH_2026`).

---

### 🎉 Summary & Next Steps
You have completed the model training and evaluation walkthrough! The exported weights in `MODELS/yolov8n.pt` are now active.

To launch the full-stack AquaSentinel application with the React Dashboard and FastAPI backend:
1. Double-click **`start.bat`** in the project root, or
2. Run `uvicorn backend.main:app --reload` and `npm run dev` in `frontend/`.
